# AI Gateways — LiteLLM, Portkey, Kong AI Gateway, Bifrost: Interactive Visual Explorer

> A gateway sits between your apps and model providers. Core features are provider routing, fallback, retries, rate limiting, secret references, observability, guardrails. Market split in 2026: **LiteLLM** is MIT OSS with 100+ providers, OpenAI-compatible, but breaks down around ~2000 RPS (8 GB memory, cascading failures in published benchmarks); best for Python, <500 RPS, dev/prototyping. **Portkey** is control-plane-positioned (guardrails, PII redaction, jailbreak detection, audit trails), went Apache 2.0 open-source March 2026, 20-40 ms latency overhead, $49/mo production tier. **Kong AI Gateway** built on Kong Gateway — Kong's own benchmark on same 12 CPUs: 228% faster than Portkey, 859% faster than LiteLLM; $100/model/month pricing (max 5 on Plus tier); enterprise-fit if you're already on Kong. **Bifrost** (Maxim AI) — automatic retries with configurable backoff, fallback to Anthropic on OpenAI 429. **Cloudflare / Vercel AI Gateways** — managed, zero-ops, basic retry. Data residency drives the self-host decision; Portkey and Kong sit in the middle with OSS + optional managed.

Welcome to the interactive companion notebook for **AI Gateways — LiteLLM, Portkey, Kong AI Gateway, Bifrost**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""AI gateway routing + fallback simulator — stdlib Python.

Models a gateway fronting OpenAI, Anthropic, and self-hosted. Injects 429/5xx
errors per provider. Compares fallback strategies.
"""

from __future__ import annotations

from dataclasses import dataclass, field
import random

@dataclass
class Provider:
    name: str
    base_latency_ms: float
    error_rate: float
    overhead_ms: float

PROVIDERS = [
    Provider("OpenAI",       180, 0.03, 0),
    Provider("Anthropic",    220, 0.02, 0),
    Provider("Self-hosted",  100, 0.05, 0),
]


In [ ]:
GATEWAY_OVERHEAD = {
    "LiteLLM": 10,
    "Portkey": 30,
    "Kong":      5,
    "Cloudflare": 2,
}

def call_provider(p: Provider, rng: random.Random) -> tuple[bool, float]:
    if rng.random() < p.error_rate:
        return False, p.base_latency_ms * 0.3  # half-done before error
    return True, p.base_latency_ms

def simulate_fallback(gateway: str, n: int = 1000, seed: int = 7) -> dict:
    rng = random.Random(seed)
    success = 0
    total_latency = 0.0
    retries = 0
    fallback_hits = 0
    gw_ovh = GATEWAY_OVERHEAD[gateway]


In [ ]:
for _ in range(n):
        req_latency = gw_ovh
        done = False
        for attempt, p in enumerate(PROVIDERS):
            ok, ms = call_provider(p, rng)
            req_latency += ms
            if attempt > 0:
                fallback_hits += 1
            if ok:
                success += 1
                done = True
                break
            retries += 1
        total_latency += req_latency


In [ ]:
return {
        "gateway": gateway,
        "success_rate": success / n,
        "mean_latency": total_latency / n,
        "retries": retries,
        "fallback_hits": fallback_hits,
    }

def report(row: dict) -> None:
    print(f"{row['gateway']:12}  success={row['success_rate']*100:5.1f}%  "
          f"mean_latency={row['mean_latency']:6.0f}ms  "
          f"retries={row['retries']:4}  fallbacks={row['fallback_hits']:4}")


In [ ]:
def main() -> None:
    print("=" * 80)
    print("AI GATEWAY FALLBACK — 3-provider chain under error injection")
    print("=" * 80)
    header = f"{'Gateway':12}  {'Success':>7}         {'mean latency':>12}  retries  fallbacks"
    print(header)
    print("-" * len(header))
    for gw in ("LiteLLM", "Portkey", "Kong", "Cloudflare"):
        report(simulate_fallback(gw))

print("\nNotes: a single-provider target at 3% error rate → 97% success.")
    print("Two-provider fallback → 99.94% success (complement of 0.03 × 0.02).")
    print("Three-provider fallback → 99.997% success. Latency rises on fallback.")


In [ ]:
if __name__ == "__main__":
    main()
